# Correlation Plots

This file should be executed **after** running `main.ipynb`.  
It generates correlation plots for top 15 features identified by the XGBoost model.  


In [ ]:
from functions import *

# load the data
df = pd.read_csv(r"../Data Cleaning/results/merged_labeled.csv", index_col="eid")
df["Status"] = df["Status"].map({"healthy": 0, "dementia": 1, "AD": 2})

CAT_FEATURES = [
    "Age",
    "Sex",
    "Educational status",
    "Diabetes",
    "Alcohol Consumption",
    "Smoking Status",
    "Antihypertensive usage",
]
NUM_FEATURES = df.columns.difference(CAT_FEATURES + ["Status", "eid"])

MAIN_FEATURES = [
    "Age",
    "Sex",
    "Educational status",
    "Diabetes",
    "Spherical equivalent",
    "Systolic blood pressure",
    "Diastolic blood pressure",
    "Antihypertensive usage",
    "Alcohol Consumption",
    "Smoking Status",
    "BMI",
    "mRNFL thickness",
    "mGCIPL thickness",
    "Status",
]
df_main = df.loc[:, MAIN_FEATURES]
MAIN_NUM_FEATURES = df_main.columns.difference(CAT_FEATURES + ["Status", "eid"])
MAIN_CAT_FEATURES = CAT_FEATURES

In [ ]:
ldf_knn_mm = []
ldf_knn_age_matched_mm = []
ldf_randomfill_mm = []
ldf_randomfill_age_matched_mm = []

# ad
df_knn_ad_top15 = pd.read_csv(
    r"./results_combined/final/feature_importances/knn_mm - AD vs Healthy_top_features.csv"
)
df_knn_ad_agematched_top15 = pd.read_csv(
    r"./results_combined/final/feature_importances/knn_age_matched_mm - AD vs Healthy_top_features.csv"
)
shap_knn_ad = pd.read_csv(
    r"./results_combined/final/feature_importances/knn_mm - AD vs Healthy_shap_values.csv"
)
shap_knn_ad_agematched = pd.read_csv(
    r"./results_combined/final/feature_importances/knn_age_matched_mm - AD vs Healthy_shap_values.csv"
)

df_randomfill_ad_top15 = pd.read_csv(
    r"./results_combined/final/feature_importances/randomfill_mm - AD vs Healthy_top_features.csv"
)
df_randomfill_ad_agematched_top15 = pd.read_csv(
    r"./results_combined/final/feature_importances/randomfill_age_matched_mm - AD vs Healthy_top_features.csv"
)
shap_randomfill_ad = pd.read_csv(
    r"./results_combined/final/feature_importances/randomfill_mm - AD vs Healthy_shap_values.csv"
)
shap_randomfill_ad_agematched = pd.read_csv(
    r"./results_combined/final/feature_importances/randomfill_age_matched_mm - AD vs Healthy_shap_values.csv"
)

df_knn_dementia_top15 = pd.read_csv(
    r"./results_combined/final/feature_importances/knn_mm - Dementia vs Healthy_top_features.csv"
)
df_knn_dementia_agematched_top15 = pd.read_csv(
    r"./results_combined/final/feature_importances/knn_age_matched_mm - Dementia vs Healthy_top_features.csv"
)
shap_knn_dementia = pd.read_csv(
    r"./results_combined/final/feature_importances/knn_mm - Dementia vs Healthy_shap_values.csv"
)
shap_knn_dementia_agematched = pd.read_csv(
    r"./results_combined/final/feature_importances/knn_age_matched_mm - Dementia vs Healthy_shap_values.csv"
)

df_randomfill_dementia_top15 = pd.read_csv(
    r"./results_combined/final/feature_importances/randomfill_mm - Dementia vs Healthy_top_features.csv"
)
df_randomfill_dementia_agematched_top15 = pd.read_csv(
    r"./results_combined/final/feature_importances/randomfill_age_matched_mm - Dementia vs Healthy_top_features.csv"
)
shap_randomfill_dementia = pd.read_csv(
    r"./results_combined/final/feature_importances/randomfill_mm - Dementia vs Healthy_shap_values.csv"
)
shap_randomfill_dementia_agematched = pd.read_csv(
    r"./results_combined/final/feature_importances/randomfill_age_matched_mm - Dementia vs Healthy_shap_values.csv"
)

df_knn_ad_agematched_full = pd.read_csv(
    r"./results_combined/final/feature_importances/knn_age_matched_mm - AD vs Healthy_all_features.csv"
)

for i in range(1, 11):
    df_knn_mm = pd.read_csv(
        r"../Data Cleaning/results_{}/missing_matched_knn.csv".format(i),
        index_col="eid",
    )
    df_knn_age_matched_mm = pd.read_csv(
        r"../Data Cleaning/results_{}/missing_matched_age_matched_knn.csv".format(i),
        index_col="eid",
    )
    df_randomfill_mm = pd.read_csv(
        r"../Data Cleaning/results_{}/missing_matched_randomfill.csv".format(i),
        index_col="eid",
    )
    df_randomfill_age_matched_mm = pd.read_csv(
        r"../Data Cleaning/results_{}/missing_matched_age_matched_randomfill.csv".format(
            i
        ),
        index_col="eid",
    )

    ldf_knn_mm.append(df_knn_mm)
    ldf_knn_age_matched_mm.append(df_knn_age_matched_mm)
    ldf_randomfill_mm.append(df_randomfill_mm)
    ldf_randomfill_age_matched_mm.append(df_randomfill_age_matched_mm)

In [ ]:
df_knn_ad_agematched_full.to_csv(r"./agematched_full.csv")
shap_knn_ad_agematched.to_csv(r"./shap_agematched_full.csv")

In [ ]:
def create_complete_correlation_map(
    input_df, title, top_features, shap_features, status="dementia"
):

    shap_top15 = shap_features.head(15)["Feature"].tolist()
    # Keep all top_features in their original order
    combined_features = list(top_features)

    # Append shap_top15 items that are not already in top_features
    combined_features.extend([f for f in shap_top15 if f not in top_features])
    (combined_features.append("Status"),)
    top_features = combined_features
    corr_matrices = []
    for df_i in input_df:
        df = df_i.loc[:, top_features].copy()
        if status == "dementia":
            df["Status"] = df["Status"].map({"healthy": 0, "dementia": 1, "AD": 1})
            df.rename(columns={"Status": "Dementia vs NonDementia"}, inplace=True)

        else:
            filter = df["Status"] == "dementia"
            df.drop(df[filter].index, inplace=True)
            df["Status"] = df["Status"].map({"healthy": 0, "AD": 1})
            df.rename(columns={"Status": "AD vs NonAD"}, inplace=True)

        corr_matrix = df.corr(method="spearman")
        corr_matrices.append(corr_matrix)

    common_columns = corr_matrices[0].columns

    corr_stack = np.stack(
        [cm.loc[common_columns, common_columns].values for cm in corr_matrices]
    )  # (5, 15, 15)

    # Ortalama korelasyon matrisi
    mean_corr_matrix = np.mean(corr_stack, axis=0)
    mean_corr_df = pd.DataFrame(
        mean_corr_matrix, index=common_columns, columns=common_columns
    )

    fig, ax = plt.subplots()
    fig.set_size_inches(25, 23)

    sns.heatmap(
        mean_corr_df,
        cmap="coolwarm",
        annot=True,
        annot_kws={"size": 14.5, "weight": "bold"},
        fmt=".2f",
        linewidths=0.5,
        linecolor="black",
    )

    plt.xticks(rotation=90, fontsize=18, fontweight="bold")
    plt.yticks(rotation=0, fontsize=18, fontweight="bold")
    plt.tight_layout()

    path = r"./results_combined/final/correlation_results_spearman/"
    os.makedirs(path, exist_ok=True)
    plt.savefig(path + f"{title}.pdf", dpi=300)
    plt.close()

    # get the top 15 highest correlations
    corr_pairs = mean_corr_df.abs().unstack()
    corr_pairs = corr_pairs[corr_pairs < 1.0]  # remove self correlations
    top_15 = corr_pairs.sort_values(ascending=False).drop_duplicates().head(15)

    print("\nTop 15 Highest Correlations:\n")
    print(top_15)

    return top_15


def create_correlation_map(input_df, title, top_features, status="dementia"):

    top_features.append("Status")
    corr_matrices = []
    for df_i in input_df:
        df = df_i.loc[:, top_features].copy()
        if status == "dementia":
            df["Status"] = df["Status"].map({"healthy": 0, "dementia": 1, "AD": 1})
            df.rename(columns={"Status": "Dementia vs NonDementia"}, inplace=True)

        else:
            filter = df["Status"] == "dementia"
            df.drop(df[filter].index, inplace=True)
            df["Status"] = df["Status"].map({"healthy": 0, "AD": 1})
            df.rename(columns={"Status": "AD vs NonAD"}, inplace=True)

        corr_matrix = df.corr(method="spearman")
        corr_matrices.append(corr_matrix)

    common_columns = corr_matrices[0].columns

    corr_stack = np.stack(
        [cm.loc[common_columns, common_columns].values for cm in corr_matrices]
    )  # (5, 15, 15)

    # Ortalama korelasyon matrisi
    mean_corr_matrix = np.mean(corr_stack, axis=0)
    mean_corr_df = pd.DataFrame(
        mean_corr_matrix, index=common_columns, columns=common_columns
    )

    fig, ax = plt.subplots()
    fig.set_size_inches(20, 18)

    sns.heatmap(
        mean_corr_df,
        cmap="coolwarm",
        annot=True,
        annot_kws={"size": 14.5, "weight": "bold"},
        fmt=".2f",
        linewidths=0.5,
        linecolor="black",
    )

    plt.xticks(rotation=90, fontsize=18, fontweight="bold")
    plt.yticks(rotation=0, fontsize=18, fontweight="bold")
    plt.tight_layout()

    path = r"./results_combined/final/correlation_results_spearman/"
    os.makedirs(path, exist_ok=True)
    plt.savefig(path + f"{title}.pdf", dpi=300)
    plt.close()

    # get the top 15 highest correlations
    corr_pairs = mean_corr_df.abs().unstack()
    corr_pairs = corr_pairs[corr_pairs < 1.0]  # remove self correlations
    top_15 = corr_pairs.sort_values(ascending=False).drop_duplicates().head(15)

    print("\nTop 15 Highest Correlations:\n")
    print(top_15)

    return top_15


def create_delta_correlation_map(
    input_dfs1, input_dfs2, title, top_features, status="dementia"
):
    """show the delta correlation map between two different datasets"""

    top_features.append("Status")
    corr_matrices_1 = []
    corr_matrices_2 = []

    for df_i in input_dfs1:
        df = df_i.loc[:, top_features].copy()
        if status == "dementia":
            df["Status"] = df["Status"].map({"healthy": 0, "dementia": 1, "AD": 1})
            df.rename(columns={"Status": "Dementia vs NonDementia"}, inplace=True)

        else:
            filter = df["Status"] == "Dementia"
            df.drop(df[filter].index, inplace=True)
            df["Status"] = df["Status"].map({"healthy": 0, "AD": 1})
            df.rename(columns={"Status": "AD vs NonAD"}, inplace=True)

        corr_matrix = df.corr()
        corr_matrices_1.append(corr_matrix)

    for df_i in input_dfs2:
        df = df_i.loc[:, top_features].copy()
        if status == "dementia":
            df["Status"] = df["Status"].map({"healthy": 0, "dementia": 1, "AD": 1})
            df.rename(columns={"Status": "Dementia vs NonDementia"}, inplace=True)

        else:
            filter = df["Status"] == "Dementia"
            df.drop(df[filter].index, inplace=True)
            df["Status"] = df["Status"].map({"healthy": 0, "AD": 1})
            df.rename(columns={"Status": "AD vs NonAD"}, inplace=True)

        corr_matrix = df.corr()
        corr_matrices_2.append(corr_matrix)

    # get the common columns from both correlation matrices
    common_columns = corr_matrices_1[0].columns

    corr_stack_1 = np.stack(
        [cm.loc[common_columns, common_columns].values for cm in corr_matrices_1]
    )  # (5, 15, 15)
    corr_stack_2 = np.stack(
        [cm.loc[common_columns, common_columns].values for cm in corr_matrices_2]
    )  # (5, 15, 15)

    mean_corr_matrix_1 = np.mean(corr_stack_1, axis=0)
    mean_corr_matrix_2 = np.mean(corr_stack_2, axis=0)

    delta_corr_matrix = mean_corr_matrix_1 - mean_corr_matrix_2

    delta_corr_df = pd.DataFrame(
        delta_corr_matrix, index=common_columns, columns=common_columns
    )

    fig, ax = plt.subplots()
    fig.set_size_inches(20, 18)

    sns.heatmap(
        delta_corr_df,
        cmap="coolwarm",
        annot=True,
        annot_kws={"size": 14.5, "weight": "bold"},
        fmt=".2f",
        linewidths=0.5,
        linecolor="black",
    )

    plt.xticks(rotation=90, fontsize=18, fontweight="bold")
    plt.yticks(rotation=0, fontsize=18, fontweight="bold")
    plt.tight_layout()

    path = r"./results_combined/final/delta_correlation_results/"
    os.makedirs(path, exist_ok=True)
    plt.savefig(path + f"{title}.pdf", dpi=300)
    plt.close()

In [ ]:
create_correlation_map(
    ldf_knn_mm,
    "knn top 15 - AD vs CN - feature correlations",
    df_knn_ad_top15["Feature"].tolist(),
    status="ad",
)

In [ ]:
create_correlation_map(
    ldf_knn_age_matched_mm,
    "knn top 15 - Age matched - AD vs CN - feature correlations",
    df_knn_ad_agematched_top15["Feature"].tolist(),
    status="ad",
)

In [ ]:
create_correlation_map(
    ldf_randomfill_mm,
    "randomfill top 15 - AD vs CN - feature correlations",
    df_randomfill_ad_top15["Feature"].tolist(),
    status="ad",
)

In [ ]:
create_correlation_map(
    ldf_randomfill_age_matched_mm,
    "randomfill top 15 - Age matched - AD vs CN - feature correlations",
    df_randomfill_ad_agematched_top15["Feature"].tolist(),
    status="ad",
)

In [ ]:
create_correlation_map(
    ldf_knn_mm,
    "XGBoost top 15 - nonagematched with agematched features - AD vs CN - show nonmatched with age matched top 15",
    df_knn_ad_agematched_top15["Feature"].tolist(),
    status="ad",
)

In [ ]:
create_complete_correlation_map(
    ldf_knn_mm,
    "XGBoost and shap - AD vs CN",
    df_knn_ad_top15["Feature"].tolist(),
    shap_knn_ad,
    status="ad",
)

In [ ]:
create_complete_correlation_map(
    ldf_knn_age_matched_mm,
    "XGBoost and shap - age matched - AD vs CN",
    df_knn_ad_agematched_top15["Feature"].tolist(),
    shap_knn_ad_agematched,
    status="ad",
)

In [ ]:
create_delta_correlation_map(
    ldf_knn_mm,
    ldf_knn_age_matched_mm,
    "XGBoost delta top 15 - Age matched - AD vs CN - feature correlations",
    df_knn_ad_agematched_top15["Feature"].tolist(),
    status="ad",
)

# dementia

In [ ]:
create_complete_correlation_map(
    ldf_knn_mm,
    "XGBoost and shaps - Dementia vs CN",
    df_knn_dementia_top15["Feature"].tolist(),
    shap_knn_dementia,
    status="dementia",
)

In [ ]:
create_complete_correlation_map(
    ldf_knn_age_matched_mm,
    "XGBoost and shap - age matched - Dementia vs CN",
    df_knn_dementia_agematched_top15["Feature"].tolist(),
    shap_knn_dementia_agematched,
    status="dementia",
)